# 4.11 Data Visualization Techniques

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 4: Data Preprocessing & Exploratory Data Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-04-data-preprocessing-and-exploratory-data-analysis/04.11_data-visualization-techniques.ipynb)

Companion code for Section 4.11 of the book. Run the cells in order from top to bottom.

**Example 4.11.1**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
df = pd.DataFrame(data.data[:, :8], columns=data.feature_names[:8])
df['target'] = data.target
df['diagnosis'] = df['target'].map({0:'Malignant', 1:'Benign'})

sns.set_theme(style='whitegrid', palette='husl', font_scale=1.0)
fig = plt.figure(figsize=(18, 14))

# ── 1. Distribution comparison (KDE) ─────────────────────────────
ax1 = fig.add_subplot(3, 3, 1)
for diag, grp in df.groupby('diagnosis'):
    ax1.hist(grp['mean radius'], bins=25, alpha=0.6, density=True, label=diag)
    grp['mean radius'].plot.kde(ax=ax1, linewidth=2)
ax1.set_title('Distribution: Mean Radius by Diagnosis')
ax1.set_xlabel('Mean Radius'); ax1.legend()

# ── 2. Box plot comparison ─────────────────────────────────────────
ax2 = fig.add_subplot(3, 3, 2)
top4 = data.feature_names[:4]
df_melt = df[list(top4)+['diagnosis']].melt(id_vars='diagnosis')
# Normalise for comparability
df_norm = df[list(top4)].apply(lambda x: (x-x.mean())/x.std())
df_norm['diagnosis'] = df['diagnosis']
df_norm_melt = df_norm.melt(id_vars='diagnosis', var_name='feature', value_name='z-score')
sns.boxplot(data=df_norm_melt, x='feature', y='z-score', hue='diagnosis',
           ax=ax2, palette={'Malignant':'#E53935','Benign':'#1976D2'})
ax2.set_title('Feature Comparison (Z-scores)'); ax2.set_xticklabels(ax2.get_xticklabels(),rotation=30)

# ── 3. Scatter: two features coloured by class ────────────────────
ax3 = fig.add_subplot(3, 3, 3)
colors = df['target'].map({0:'#E53935', 1:'#1976D2'})
ax3.scatter(df['mean radius'], df['mean texture'], c=colors, alpha=0.6, s=20, edgecolors='none')
ax3.set_xlabel('Mean Radius'); ax3.set_ylabel('Mean Texture')
ax3.set_title('Scatter: Radius vs Texture')
from matplotlib.patches import Patch
ax3.legend(handles=[Patch(color='#E53935',label='Malignant'), Patch(color='#1976D2',label='Benign')])

# ── 4. Correlation heatmap ────────────────────────────────────────
ax4 = fig.add_subplot(3, 3, 4)
corr = df.drop(['target','diagnosis'],axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, ax=ax4, mask=mask, annot=True, fmt='.2f',
           cmap='RdYlGn', center=0, linewidths=0.3, annot_kws={'size':8})
ax4.set_title('Feature Correlation Matrix')

# ── 5. Class balance plot ─────────────────────────────────────────
ax5 = fig.add_subplot(3, 3, 5)
counts = df['diagnosis'].value_counts()
bars = ax5.bar(counts.index, counts.values, color=['#E53935','#1976D2'], edgecolor='white')
for bar, count in zip(bars, counts.values):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
            f'{count}\n({count/len(df):.1%})', ha='center', va='bottom', fontweight='bold')
ax5.set_title('Class Distribution'); ax5.set_ylabel('Count')

# ── 6. Q-Q plot - normality check ─────────────────────────────────
ax6 = fig.add_subplot(3, 3, 6)
stats.probplot(df['mean radius'], dist='norm', plot=ax6)
ax6.set_title('Q-Q Plot: Mean Radius (Normality Check)')

# ── 7. Violin plot ────────────────────────────────────────────────
ax7 = fig.add_subplot(3, 3, 7)
sns.violinplot(data=df, x='diagnosis', y='mean area',
              palette={'Malignant':'#E53935','Benign':'#1976D2'},
              ax=ax7, inner='quartile', cut=0)
ax7.set_title('Violin: Mean Area by Diagnosis')

# ── 8. Feature importance (correlation with target) ───────────────
ax8 = fig.add_subplot(3, 3, 8)
feat_corr = df.drop(['target','diagnosis'],axis=1).corrwith(df['target']).sort_values()
colors_fi = ['#1976D2' if v > 0 else '#E53935' for v in feat_corr]
ax8.barh(feat_corr.index, feat_corr.values, color=colors_fi, edgecolor='white')
ax8.axvline(x=0, color='black', linewidth=0.8)
ax8.set_xlabel('Pearson r with target'); ax8.set_title('Feature-Target Correlation')

# ── 9. Pair plot subset (separate) ────────────────────────────────
ax9 = fig.add_subplot(3, 3, 9)
sns.scatterplot(data=df, x='mean concave points', y='worst radius',
               hue='diagnosis', palette={'Malignant':'#E53935','Benign':'#1976D2'},
               alpha=0.7, s=30, ax=ax9)
ax9.set_title('Most Correlated Feature Pair')

plt.tight_layout(pad=2.0)
plt.savefig('eda_visualization_suite.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA visualization suite saved.')